# 02 V2 模型评测

**目标**：调用模型 API，收集 V2 数据集的回答，输出 `results/v2/raw/raw_results.csv`

**建议顺序**：
1. 先跑 `deepseek` 验证 V2 元数据和断点续跑
2. 再扩展到 `kimi / qwen`
3. 完成后进入 `notebooks/v2/03_analysis_visualization_v2.ipynb` 做变体与效率分析

⚠️ **前置条件**：已在 `.env` 文件中配置 API Key，且先完成 `01_data_preparation_v2.ipynb`

In [10]:
import sys
from pathlib import Path
import importlib

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'v2' and PROJECT_ROOT.parent.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parents[1]
elif PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import os
import pandas as pd
import yaml
from dotenv import load_dotenv

load_dotenv(PROJECT_ROOT / '.env')

import src.eval_runner as eval_runner_module
import src.eval_runner_v2 as eval_runner_v2_module

importlib.reload(eval_runner_module)
importlib.reload(eval_runner_v2_module)

from src.eval_runner import call_model, get_client
from src.eval_runner_v2 import run_eval_bundle_v2, run_eval_v2

config = yaml.safe_load(open(PROJECT_ROOT / 'configs/eval_config_v2.yaml', encoding='utf-8'))
config['data']['processed_dir'] = str(PROJECT_ROOT / config['data']['processed_dir'])
config['results']['raw_dir'] = str(PROJECT_ROOT / config['results']['raw_dir'])
config['results']['processed_dir'] = str(PROJECT_ROOT / config['results']['processed_dir'])
config['results']['figures_dir'] = str(PROJECT_ROOT / config['results']['figures_dir'])

print('V2 配置加载成功')
print(f'PROJECT_ROOT: {PROJECT_ROOT}')
print(f"V2 数据目录: {config['data']['processed_dir']}")
print(f"V2 结果目录: {config['results']['raw_dir']}")

V2 配置加载成功
PROJECT_ROOT: /Users/melody/Desktop/Eval
V2 数据目录: /Users/melody/Desktop/Eval/data/processed/v2
V2 结果目录: /Users/melody/Desktop/Eval/results/v2/raw


## Step 1：检查 API Key 配置

In [5]:
key_map = {
    'deepseek': 'DEEPSEEK_API_KEY',
    'kimi':     'MOONSHOT_API_KEY',  # moonshot-v1-128k；KIMI_API_KEY 为 Coding 专用
    'qwen':     'DASHSCOPE_API_KEY',
}
print('API Key 状态：')
for model, env_var in key_map.items():
    val = os.getenv(env_var, '')
    status = '✅ 已配置' if val and not val.startswith('sk-xxx') else '❌ 未配置'
    masked = (val[:8] + '...' + val[-4:]) if len(val) > 12 else '（空）'
    print(f'  {model:12s} ({env_var}): {status}  {masked}')

API Key 状态：
  deepseek     (DEEPSEEK_API_KEY): ✅ 已配置  sk-5cfdc...a922
  kimi         (MOONSHOT_API_KEY): ✅ 已配置  sk-sExnE...ifpB
  qwen         (DASHSCOPE_API_KEY): ✅ 已配置  sk-a1bce...10e4


## Step 2：单条冒烟测试（验证 API 连通性）

In [6]:
TEST_MODEL = 'deepseek'  # 改为你想测试的模型

try:
    client, model_name = get_client(TEST_MODEL, config)
    test_context = '董事会批准 2025 年资本开支上限为 18.6 亿元，财务部草案曾讨论 17.9 亿元。'
    test_question = '董事会批准的资本开支上限是多少？'
    
    response, prompt_tokens, completion_tokens, cached_tokens, latency, error = call_model(
        client, model_name,
        context=test_context,
        question=test_question,
        max_tokens=64,
    )
    print(f'✅ [{TEST_MODEL}] V2 连通性测试通过')
    print(f'   问题: {test_question}')
    print(f'   回答: {response}')
    print(f'   Prompt: {prompt_tokens}  Completion: {completion_tokens}  Cache: {cached_tokens}  Latency: {latency:.2f}s')
    if error:
        print(f'   Error: {error}')
except Exception as e:
    print(f'❌ 连通性测试失败: {e}')

✅ [deepseek] V2 连通性测试通过
   问题: 董事会批准的资本开支上限是多少？
   回答: 18.6亿元
   Prompt: 101  Completion: 4  Cache: 0  Latency: 0.81s


## Step 3：批量评测

调整下方 `MODEL_KEYS` 和 `MAX_SAMPLES` 控制 V2 评测范围：

In [13]:
# ── 可调参数 ──────────────────────────────────
MODEL_KEYS = ['deepseek', 'kimi', 'qwen']
MAX_SAMPLES = None              # None = 跑全量（约 350 条/模型）
RESUME = True                   # True = 断点续跑，按 sample_id 跳过已有结果
# ────────────────────────────────────────────

dataset_path = Path(config['data']['processed_dir']) / 'niah_dataset.jsonl'

if not dataset_path.exists():
    print('⚠️  V2 数据集不存在，请先运行 notebooks/v2/01_data_preparation_v2.ipynb')
else:
    df = run_eval_v2(
        dataset_path=str(dataset_path),
        model_keys=MODEL_KEYS,
        config=config,
        output_dir=config['results']['raw_dir'],
        max_samples=MAX_SAMPLES,
        resume=RESUME,
    )
    print(f'\n完成！共 {len(df)} 条 V2 结果')
    df.head(5)

📂 加载 350 条 V2 样本，来自: /Users/melody/Desktop/Eval/data/processed/v2/niah_dataset.jsonl
   已有 1062 条结果，启用 V2 断点续跑

🚀 [V2:deepseek] deepseek-chat — RPM 限制: 60


v2-deepseek: 100%|██████████| 350/350 [00:00<00:00, 1671989.07req/s]



🚀 [V2:kimi] moonshot-v1-128k — RPM 限制: 20


v2-kimi: 100%|██████████| 350/350 [00:00<00:00, 2410519.54req/s]



🚀 [V2:qwen] qwen-long — RPM 限制: 60


v2-qwen: 100%|██████████| 350/350 [00:00<00:00, 2612111.03req/s]

ℹ️  无新结果（所有样本已处理或无可用 API Key）

完成！共 1062 条 V2 结果


## Step 4：快速预览结果

In [8]:
results_path = Path(config['results']['raw_dir']) / 'raw_results.csv'

if results_path.exists():
    df = pd.read_csv(results_path)
    print(f'结果统计：{len(df)} 条')
    print('\n模型分布：')
    print(df['model'].value_counts().to_string())
    if 'variant' in df.columns:
        print('\n变体分布：')
        print(df['variant'].value_counts().to_string())
    if 'domain' in df.columns:
        print('\n领域分布：')
        print(df['domain'].value_counts().to_string())
    print('\n前 5 条样本：')
    display(df[[
        'model', 'variant', 'domain', 'context_length', 'depth_pct',
        'num_needles', 'expected_answer', 'model_response',
        'completion_tokens', 'latency_s'
    ]].head(5))

结果统计：1050 条

模型分布：
model
deepseek    350
kimi        350
qwen        350

变体分布：
variant
multi_key             360
style_aligned         348
numeric_confusable    342

领域分布：
domain
finance          330
manufacturing    264
healthcare       213
retail           129
logistics        114

前 5 条样本：


,model,variant,domain,context_length,depth_pct,num_needles,expected_answer,model_response,completion_tokens,latency_s
0,deepseek,multi_key,finance,2000,0,3,9.4亿元,9.4亿元,4,0.53
1,deepseek,style_aligned,manufacturing,2000,0,1,98.2%,98.2%,4,0.48
2,deepseek,numeric_confusable,logistics,2000,0,1,92.1%,92.1%,4,0.58
3,deepseek,multi_key,finance,2000,0,3,9.4亿元,9.4亿元,4,0.84
4,deepseek,style_aligned,manufacturing,2000,0,1,98.2%,98.2%,4,0.51


## Step 5：效率摘要（长度 / 输出 token / 单位正确率成本）

In [9]:
from src.metrics_v2 import score_results_v2, summarize_v2

if results_path.exists():
    df = pd.read_csv(results_path)
    scored = score_results_v2(df)
    summary = summarize_v2(scored)
    print('V2 效率摘要（按模型）：\n')
    print(summary[[
        'model', 'n', 'contains_pct', 'contains_ci_low_pct', 'contains_ci_high_pct',
        'avg_response_chars', 'avg_completion_tokens', 'total_cost_cny',
        'cost_per_contains_hit_cny',
    ]].to_string(index=False))

V2 效率摘要（按模型）：

   model   n  contains_pct  contains_ci_low_pct  contains_ci_high_pct  avg_response_chars  avg_completion_tokens  total_cost_cny  cost_per_contains_hit_cny
deepseek 350          90.6                 87.1                  93.2                 6.1                    4.8          1.8211                     0.0057
    kimi 350          63.1                 58.0                  68.0                 4.5                    4.5         86.6157                     0.3919
    qwen 350          81.4                 77.0                  85.2                 6.0                    5.5         11.3116                     0.0397


## ✅ V2 评测完成

下一步：打开 `notebooks/v2/03_analysis_visualization_v2.ipynb` 做变体、效率和稳定性分析